In [ ]:
# uncomment these and run this cell if needed
!pip install evaluate
!pip install huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.8 MB/s eta 0:00:00


In [ ]:
from datasets import load_dataset
from transformers import AutoModelForSequenceClassification
from transformers import AutoTokenizer
from transformers import get_scheduler
from transformers import Trainer, TrainingArguments
import evaluate
import numpy as np

In [ ]:
#initalize variables
header1 = "Question: "
header2 = "Passage: "
header3 = "Answer: "

column_names1 = "question"
column_names2 = "passage"
column_names3 = "answer"

def set_header1(header_value):
  global header1
  header1=header_value

def set_header2(header_value):
  global header2
  header2=header_value

def set_header3(header_value):
  global header3
  header3=header_value

def set_column_names1(column_names):
  global column_names1
  column_names1=column_names

def set_column_names2(column_names):
  global column_names2
  column_names2=column_names

def set_column_names3(column_names):
  global column_names3
  column_names3=column_names

def set_master_columns(col1, col2, col3, header_val1, header_val2, header_val3):
  global column_names1, column_names2, column_names3, header1, header2, header3
  column_names1=col1
  column_names2=col2
  column_names3=col3

  header1=header_val1
  header2=header_val2
  header3=header_val3

In [ ]:
# prepping the dataset
raw_datasets = load_dataset("flowaicom/HaluEval")#https://huggingface.co/datasets/flowaicom/HaluEval

def label_map(example):
    if example["label"] == "PASS" or example["label"] == "1" or example["label"] == 1:
        example["label"] = 1
    else:
        example["label"] = 0
    return example

raw_datasets = raw_datasets.map(label_map)

tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

def tokenize_function(examples):
    global header1, header2, header3, column_names1, column_names2, column_names3
    combined = [
        (header1 + q, header2 + p + header3 + a)
        for q, p, a in zip(examples[column_names1], examples[column_names2], examples[column_names3])
    ]

    return tokenizer(
        [c[0] for c in combined],
        [c[1] for c in combined],
        padding="max_length",
        truncation=True
    )
#[SEP] is a special token used in BERT to separate different parts of the input sequence, such as the question, context, and answer, allowing the model to distinguish between them.
#[CLS] start of input
#tokenizer does this automatically for us



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/3.50M [00:00<?, ?B/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
set_master_columns("question", "passage", "answer", "Question: ", "Passage: ", "Answer: ")

tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)

#tokenized_datasets = tokenized_datasets.remove_columns([
#    "id", "passage", "question", "answer", "source_ds", "score"
#])

#Bert will only have input_ids, atteniton_mask, typetype_id and label left

# split dataset since we do not have an training data only test data
split_datasets = tokenized_datasets["test"].train_test_split(test_size=0.2, seed=42)
#train_test_split = “don't test on what you trained on”

small_train_dataset = split_datasets["train"].shuffle(seed=42).select(range(400))
small_eval_dataset = split_datasets["test"].shuffle(seed=42).select(range(100))

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

In [ ]:
#setting up the model

#default
model = AutoModelForSequenceClassification.from_pretrained("bert-base-cased", num_labels=2)

accuracy = evaluate.load("accuracy")
precision = evaluate.load("precision")
recall = evaluate.load("recall")
f1 = evaluate.load("f1")

#added average binary because it tells the model to Treat this as a binary classification problem and focus on the positive class (1)
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy.compute(predictions=predictions, references=labels)["accuracy"],
        "precision": precision.compute(predictions=predictions, references=labels, average="binary")["precision"],
        "recall": recall.compute(predictions=predictions, references=labels, average="binary")["recall"],
        "f1": f1.compute(predictions=predictions, references=labels, average="binary")["f1"],
    }

training_args = TrainingArguments(eval_strategy="epoch", num_train_epochs = 8, learning_rate = 2e-5)

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
#train
trainer = Trainer(
    model=model, args=training_args, train_dataset=small_train_dataset, eval_dataset=small_eval_dataset, compute_metrics=compute_metrics,
)
trainer.train()
trainer.evaluate()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,No log,0.562428,0.490000,0.490000,1.000000,0.657718
2,No log,0.392054,0.920000,0.859649,1.000000,0.924528
3,No log,0.185366,0.970000,0.942308,1.000000,0.970297
4,No log,0.243536,0.960000,0.924528,1.000000,0.960784
5,No log,0.237401,0.970000,0.942308,1.000000,0.970297
6,No log,0.242043,0.970000,0.942308,1.000000,0.970297
7,No log,0.252477,0.960000,0.924528,1.000000,0.960784
8,No log,0.264530,0.960000,0.924528,1.000000,0.960784


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'eval_loss': 0.2645304203033447,
 'eval_accuracy': 0.96,
 'eval_precision': 0.9245283018867925,
 'eval_recall': 1.0,
 'eval_f1': 0.9607843137254902,
 'eval_runtime': 0.7692,
 'eval_samples_per_second': 129.998,
 'eval_steps_per_second': 16.9,
 'epoch': 8.0}

In [ ]:
#switch Question and passage
set_master_columns("passage", "question", "answer", "Passage: ", "Question: ", "Answer: ")


tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)

# split dataset since we do not have an training data only test data
split_datasets = tokenized_datasets["test"].train_test_split(test_size=0.2, seed=42)
#train_test_split = “don't test on what you trained on”

small_train_dataset = split_datasets["train"].shuffle(seed=42).select(range(400))
small_eval_dataset = split_datasets["test"].shuffle(seed=42).select(range(100))


Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

In [ ]:
#setting up the model

#default
model = AutoModelForSequenceClassification.from_pretrained("bert-base-cased", num_labels=2)

training_args = TrainingArguments(eval_strategy="epoch", num_train_epochs = 8, learning_rate = 2e-5)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
#train
trainer = Trainer(
    model=model, args=training_args, train_dataset=small_train_dataset, eval_dataset=small_eval_dataset, compute_metrics=compute_metrics,
)
trainer.train()
trainer.evaluate()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,No log,0.471460,0.930000,0.875000,1.000000,0.933333
2,No log,0.130566,0.970000,0.942308,1.000000,0.970297
3,No log,0.091582,0.980000,0.960784,1.000000,0.980000
4,No log,0.149205,0.970000,0.942308,1.000000,0.970297
5,No log,0.181031,0.970000,0.942308,1.000000,0.970297
6,No log,0.188092,0.970000,0.942308,1.000000,0.970297
7,No log,0.201620,0.970000,0.942308,1.000000,0.970297
8,No log,0.203308,0.970000,0.942308,1.000000,0.970297


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'eval_loss': 0.20330801606178284,
 'eval_accuracy': 0.97,
 'eval_precision': 0.9423076923076923,
 'eval_recall': 1.0,
 'eval_f1': 0.9702970297029703,
 'eval_runtime': 0.7721,
 'eval_samples_per_second': 129.511,
 'eval_steps_per_second': 16.836,
 'epoch': 8.0}

In [ ]:

ds = load_dataset("Cleanlab/FinQA-hallucination-detection") #https://huggingface.co/datasets/Cleanlab/FinQA-hallucination-detection

README.md: 0.00B [00:00, ?B/s]

finqa-hallucination-detection.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/1657 [00:00<?, ? examples/s]

In [ ]:
#switch to a different data set for evalute
set_master_columns("passage", "question", "answer", "Passage: ", "Question: ", "Answer: ")

cols_to_keep = ["question", "passage", "answer", "label"]

raw_eval = ds.rename_columns({
    "query": "question",
    "context": "passage",
    "llm_response": "answer",
    "is_correct": "label"
})

raw_eval = raw_eval.remove_columns(
    [col for col in raw_eval["train"].column_names if col not in cols_to_keep]
)

def truncate_context(example):
    example["passage"] = str(example["passage"])[:200]
    example["question"] = str(example["question"])[:100]
    return example


raw_eval = raw_eval.map(truncate_context)
from datasets import concatenate_datasets

# split by label
label_0 = raw_eval["train"].filter(lambda x: x["label"] == 0)
label_1 = raw_eval["train"].filter(lambda x: x["label"] == 1)

# get smallest class size
min_size = min(len(label_0), len(label_1))
print(len(label_0))
print(len(label_1))

# downsample both to same size
label_0 = label_0.shuffle(seed=42).select(range(min_size))
label_1 = label_1.shuffle(seed=42).select(range(min_size))

# combine + shuffle
balanced_eval = concatenate_datasets([label_0, label_1]).shuffle(seed=42)

balanced_eval=balanced_eval.map(tokenize_function, batched=True)


small_train_dataset = split_datasets["train"].shuffle(seed=42).select(range(400))
small_eval_dataset = balanced_eval.shuffle(seed=42).select(range(100))

Map:   0%|          | 0/1657 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1657 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1657 [00:00<?, ? examples/s]

239
1418


Map:   0%|          | 0/478 [00:00<?, ? examples/s]

In [ ]:
#setting up the model

#default
model = AutoModelForSequenceClassification.from_pretrained("bert-base-cased", num_labels=2)

training_args = TrainingArguments(eval_strategy="epoch", num_train_epochs = 8, learning_rate = 2e-5)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
#train
trainer = Trainer(
    model=model, args=training_args, train_dataset=small_train_dataset, eval_dataset=small_eval_dataset, compute_metrics=compute_metrics,
)
trainer.train()
trainer.evaluate()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,No log,0.750234,0.410000,0.406593,0.880952,0.556391
2,No log,1.120416,0.430000,0.421053,0.952381,0.583942
3,No log,1.792265,0.440000,0.423913,0.928571,0.582090
4,No log,2.224715,0.420000,0.414894,0.928571,0.573529
5,No log,2.758746,0.420000,0.414894,0.928571,0.573529
6,No log,2.779073,0.450000,0.428571,0.928571,0.586466
7,No log,3.121059,0.420000,0.414894,0.928571,0.573529
8,No log,3.166686,0.420000,0.414894,0.928571,0.573529


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'eval_loss': 3.1666862964630127,
 'eval_accuracy': 0.42,
 'eval_precision': 0.4148936170212766,
 'eval_recall': 0.9285714285714286,
 'eval_f1': 0.5735294117647058,
 'eval_runtime': 0.7684,
 'eval_samples_per_second': 130.145,
 'eval_steps_per_second': 16.919,
 'epoch': 8.0}